## Limpieza y Estretagias de Imputación

### Ejercicio 1. Limpieza e imputación de un conjunto de datos mixto

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.DataFrame({
    'Nombre': ['Ana', 'Luis', 'Pedro', None, 'Marta', 'Luis', 'Sofía'],
    'Edad': [25, np.nan, 35, 29, -5, 25, None],
    'Ciudad': ['Bogotá', 'Medellín', None, 'Medellín', 'Cali', 'Bogotá', 'Cali'],
    'Ingreso': [3500, 4800, np.nan, 5200, 5100, np.nan, 4700],
    'FechaIngreso': ['2023-01-01', '2023-01-05', None, '2023-01-10', '2023-01-12', None, '2023-01-15']
    })

#### Detección de valores faltantes

In [2]:
# Conteo de valores faltantes ordenado de mayor a menor
faltantes = df.isnull().sum().sort_values(ascending=False)

print("\nConteo de valores faltantes por variable:")
print(faltantes[faltantes > 0])  # Solo muestra variables con al menos un valor faltante


Conteo de valores faltantes por variable:
Edad            2
FechaIngreso    2
Ingreso         2
Nombre          1
Ciudad          1
dtype: int64


#### Eliminación de duplicados

No se evidencia valores duplicados, ya que Luis, el nombre que se repite, no es de la misma ciudad

#### Corrección de errores tipográficos y de codificación

las ciudades que estan, son iguales en tipografía 

#### Normalización de formatos

todas las fechas cumplen con el formato año-mes-día

In [3]:
df['FechaIngreso'] = pd.to_datetime(df['FechaIngreso'], errors='coerce')
print(df['FechaIngreso'])

0   2023-01-01
1   2023-01-05
2          NaT
3   2023-01-10
4   2023-01-12
5          NaT
6   2023-01-15
Name: FechaIngreso, dtype: datetime64[ns]


#### Selección de estrategia de imputación

In [4]:
df['Ingreso'] = df['Ingreso'].fillna(df['Ingreso'].median())
print(df['Ingreso'])

0    3500.0
1    4800.0
2    4800.0
3    5200.0
4    5100.0
5    4800.0
6    4700.0
Name: Ingreso, dtype: float64


In [5]:
df['Nombre'].fillna(df['Nombre'].mode()[0])
print(df['Nombre'])

0      Ana
1     Luis
2    Pedro
3     None
4    Marta
5     Luis
6    Sofía
Name: Nombre, dtype: object


In [6]:
df['Edad'] = df['Edad'].apply(lambda x: np.nan if x < 0 else x)
df['Edad'] = df['Edad'].fillna(df['Edad'].median())
print(df['Edad'])

0    25.0
1    27.0
2    35.0
3    29.0
4    27.0
5    25.0
6    27.0
Name: Edad, dtype: float64


In [7]:
df['Ciudad'].fillna(df['Ciudad'].mode()[0])
print(df['Ciudad'])

0      Bogotá
1    Medellín
2        None
3    Medellín
4        Cali
5      Bogotá
6        Cali
Name: Ciudad, dtype: object


In [8]:
df['FechaIngreso'].fillna(df['FechaIngreso'].mode()[0])
print(df['FechaIngreso'])

0   2023-01-01
1   2023-01-05
2          NaT
3   2023-01-10
4   2023-01-12
5          NaT
6   2023-01-15
Name: FechaIngreso, dtype: datetime64[ns]


### Ejercicio 2. Limpieza de duplicados

In [9]:
df = pd.DataFrame({
    'ID': [101, 102, 102, 103, 104, 104, 104],
    'Nombre': ['Ana', 'Luis', 'Luis', 'Marta', 'Carlos', 'Carlos', 'Carlos'],
    'Edad': [25, 30, 30, 29, 40, 40, 41],
    'Ciudad': ['Bogotá', 'Cali', 'Cali', 'Medellín', 'Cali', 'Cali', 'Cali'],
    'FechaRegistro': ['2023-01-01', '2023-01-05', '2023-01-05', '2023-01-10', 
                    '2023-01-15', '2023-01-15', '2023-01-16']
})

En el mismo notebook anterior, para el nuevo dataframe df, responde a las siguientes preguntas (utilizando python):

¿Cuál es el total de registros originales?


In [10]:
# Número de registros y variables
print(f"Número de registros: {df.shape[0]}")
print(f"Número de variables: {df.shape[1]}")

Número de registros: 7
Número de variables: 5


¿Cuáles y cuántos son los duplicados exactos?

In [11]:
df[df.duplicated(keep=False)]

,ID,Nombre,Edad,Ciudad,FechaRegistro
1,102,Luis,30,Cali,2023-01-05
2,102,Luis,30,Cali,2023-01-05
4,104,Carlos,40,Cali,2023-01-15
5,104,Carlos,40,Cali,2023-01-15


¿Cuáles y cuántos son los duplicados por varias columnas?

In [12]:
df['Nombre'].value_counts()

Nombre
Carlos    3
Luis      2
Ana       1
Marta     1
Name: count, dtype: int64

In [13]:
df['Edad'].value_counts()

Edad
30    2
40    2
25    1
29    1
41    1
Name: count, dtype: int64

In [14]:
df['Ciudad'].value_counts()

Ciudad
Cali        5
Bogotá      1
Medellín    1
Name: count, dtype: int64

¿Cuántos registros debes eliminar?

Se deben eliminar 2 registros

¿Cuántos registros quedan después de la limpieza?

In [15]:
df = df.drop_duplicates()
print(f"Número de registros: {df.shape[0]}")

Número de registros: 5


### Ejercicio 3. Corrección de errores tipográficos o de codificación

In [16]:
df = pd.DataFrame({
    'Ciudad': ['bogota', 'Bogotá', 'BOGOTA', 'bogotá', 'bogata', 'Bógota', 'BogoTa', 'Cali', 'calí', 'medellín', 'medellin']
})

In [17]:
from difflib import get_close_matches
get_close_matches('Medelin', ['Medellín', 'Bogotá', 'Cali'])

['Medellín']

In [18]:
# Lista de nombres correctos/estandarizados
ciudades_validas = ['bogota', 'cali', 'medellin']

# Función para estandarizar usando get_close_matches
def corregir_ciudad(nombre):
    nombre = nombre.strip().lower()
    coincidencias = get_close_matches(nombre, [c.lower() for c in ciudades_validas], n=1, cutoff=0.6)
    if coincidencias:
        for c in ciudades_validas:
            if c.lower() == coincidencias[0]:
                return c
    else:
        return nombre.capitalize()

# Aplicar corrección al DataFrame
df['Ciudad'] = df['Ciudad'].apply(corregir_ciudad)

# Mostrar resultado
print(df)


      Ciudad
0     bogota
1     bogota
2     bogota
3     bogota
4     bogota
5     bogota
6     bogota
7       cali
8       cali
9   medellin
10  medellin
